# DSA Week 12 -- Dynamic Programming

**Course:** Data Structures & Algorithms
**Session:** 3 hours
**Prerequisites:** Weeks 1-11
**Focus:** Memoization, DP patterns, optimization

## Learning Objectives

1. Explain what dynamic programming is and when to use it
2. Implement memoization (top-down DP)
3. Implement tabulation (bottom-up DP)
4. Recognize the two requirements for DP: overlapping subproblems + optimal substructure
5. Apply DP to practical optimization problems

## The Big Idea

Dynamic Programming solves problems by breaking them into **overlapping subproblems**
and **remembering** (caching) the results so you never solve the same subproblem twice.

```
Fibonacci WITHOUT memoization -- O(2^n):

                    fib(5)
                  /        \
             fib(4)        fib(3)
            /     \        /    \
        fib(3)  fib(2)  fib(2)  fib(1)
        /   \    ...     ...
    fib(2) fib(1)

  fib(3) is computed 2 times!
  fib(2) is computed 3 times!
  Total calls: 15 (grows exponentially)

Fibonacci WITH memoization -- O(n):

  fib(5) -> fib(4) -> fib(3) -> fib(2) -> fib(1) -> fib(0)
                                 (cache)   (cache)   (cache)
  fib(3) = cached!  fib(2) = cached!
  Total calls: 6 (grows linearly)
```

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: The Fibonacci Example -- Slow vs Fast

In [ ]:
import time

# WITHOUT memoization -- O(2^n) EXPONENTIAL
call_count_slow = 0
def fib_slow(n):
    global call_count_slow
    call_count_slow += 1
    if n <= 1:
        return n
    return fib_slow(n - 1) + fib_slow(n - 2)

# WITH memoization -- O(n) LINEAR
call_count_memo = 0
def fib_memo(n, cache={}):
    global call_count_memo
    call_count_memo += 1
    if n in cache:
        return cache[n]
    if n <= 1:
        return n
    cache[n] = fib_memo(n - 1, cache) + fib_memo(n - 2, cache)
    return cache[n]

# Compare
print("=== Fibonacci: Naive vs Memoized ===")
print()

for target in [10, 20, 30]:
    call_count_slow = 0
    call_count_memo = 0

    start = time.time()
    result_slow = fib_slow(target)
    t_slow = time.time() - start

    fib_memo_cache = {}
    start = time.time()
    result_memo = fib_memo(target, fib_memo_cache)
    t_memo = time.time() - start

    print("  fib(" + str(target) + ") = " + str(result_slow))
    print("    Naive: " + str(call_count_slow) + " calls, " + "{:.4f}".format(t_slow) + "s")
    print("    Memo:  " + str(call_count_memo) + " calls, " + "{:.6f}".format(t_memo) + "s")
    if t_memo > 0:
        print("    Speedup: " + "{:.0f}".format(t_slow / t_memo) + "x")
    print()

**Expected Output:**
```
=== Fibonacci: Naive vs Memoized ===

  fib(10) = 55
    Naive: 177 calls, 0.0001s
    Memo:  19 calls, 0.000001s
    Speedup: 100x

  fib(20) = 6765
    Naive: 21891 calls, 0.0050s
    Memo:  39 calls, 0.000001s
    Speedup: 5000x

  fib(30) = 832040
    Naive: 2692537 calls, 0.5000s
    Memo:  59 calls, 0.000001s
    Speedup: 500000x
```

The naive version's call count **doubles** with every increase of 1.
The memoized version grows **linearly**.

---
## Part 2: The Easy Way -- `@functools.lru_cache`

Python has built-in memoization. Just add the decorator:

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib_cached(n):
    """Fibonacci with automatic memoization."""
    if n <= 1:
        return n
    return fib_cached(n - 1) + fib_cached(n - 2)

# This can handle MUCH larger values
print("fib(50)  = " + str(fib_cached(50)))
print("fib(100) = " + str(fib_cached(100)))
print("fib(200) = " + str(fib_cached(200)))
print()
print("Cache info:", fib_cached.cache_info())

---
## Part 3: Bottom-Up DP (Tabulation)

Instead of top-down recursion with caching, you can build the solution
from the bottom up using a table:

In [ ]:
def fib_table(n):
    """Fibonacci using bottom-up DP. O(n) time, O(n) space."""
    if n <= 1:
        return n
    table = [0] * (n + 1)
    table[0] = 0
    table[1] = 1
    for i in range(2, n + 1):
        table[i] = table[i - 1] + table[i - 2]
    return table[n]

def fib_optimized(n):
    """Fibonacci with O(1) space -- only need last two values."""
    if n <= 1:
        return n
    prev2, prev1 = 0, 1
    for i in range(2, n + 1):
        current = prev1 + prev2
        prev2 = prev1
        prev1 = current
    return prev1

print("Bottom-up table:     fib(100) = " + str(fib_table(100)))
print("Space-optimized:     fib(100) = " + str(fib_optimized(100)))

---
## Part 4: Practical DP -- Maximum Subarray Sum (Kadane's Algorithm)

This is a classic DP problem with real applications: find the contiguous
subarray with the largest sum.

```
Data: [-2, 1, -3, 4, -1, 2, 1, -5, 4]

Brute force: try all O(n^2) subarrays -- SLOW
DP (Kadane): scan once, keep running max -- O(n)

Trace:
  i=0: val=-2, current=max(-2, 0+(-2))=-2, best=-2
  i=1: val= 1, current=max(1, -2+1)=1,     best=1
  i=2: val=-3, current=max(-3, 1+(-3))=-2,  best=1
  i=3: val= 4, current=max(4, -2+4)=4,      best=4
  i=4: val=-1, current=max(-1, 4+(-1))=3,   best=4
  i=5: val= 2, current=max(2, 3+2)=5,       best=5
  i=6: val= 1, current=max(1, 5+1)=6,       best=6   <-- answer!
  i=7: val=-5, current=max(-5, 6+(-5))=1,   best=6
  i=8: val= 4, current=max(4, 1+4)=5,       best=6

Answer: 6 (subarray [4, -1, 2, 1])
```

In [ ]:
def max_subarray_brute(arr):
    """Brute force: try all subarrays. O(n^2)."""
    n = len(arr)
    best = float("-inf")
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += arr[j]
            best = max(best, current_sum)
    return best

def max_subarray_dp(arr):
    """Kadane's algorithm (DP). O(n)."""
    best = current = arr[0]
    for x in arr[1:]:
        current = max(x, current + x)
        best = max(best, current)
    return best

# Test
data = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
print("Data: " + str(data))
print("Brute force result: " + str(max_subarray_brute(data)))
print("DP result:          " + str(max_subarray_dp(data)))
print()

# Timing comparison
import timeit
import random
random.seed(42)
big_data = [random.randint(-10, 10) for _ in range(5000)]

t_brute = timeit.timeit(lambda: max_subarray_brute(big_data), number=5) / 5
t_dp = timeit.timeit(lambda: max_subarray_dp(big_data), number=5) / 5

print("Benchmark (n=5000):")
print("  Brute force: " + "{:.3f}".format(t_brute) + "s")
print("  DP (Kadane): " + "{:.6f}".format(t_dp) + "s")
print("  Speedup:     " + "{:.0f}".format(t_brute / t_dp) + "x")

---
## When to Use DP

DP works when your problem has:
1. **Overlapping subproblems** -- same smaller problem solved multiple times
2. **Optimal substructure** -- optimal solution contains optimal solutions to subproblems

| Problem | DP? | Why |
|---------|-----|-----|
| Fibonacci | YES | fib(n) depends on fib(n-1) + fib(n-2) repeatedly |
| Max subarray | YES | Current max depends on previous max |
| Shortest path (Dijkstra) | YES | Shortest to D goes through shortest to B or C |
| Sorting | NO | No overlapping subproblems |
| Finding max in list | NO | No subproblem structure |

## Common Mistakes

| Mistake | Fix |
|---------|-----|
| Not recognizing overlapping subproblems | Draw the recursion tree -- do you see repeats? |
| Mutable default argument for cache | Use `@lru_cache` instead of `cache={}` |
| Stack overflow on deep recursion | Use bottom-up tabulation instead of recursion |
| Forgetting base cases | Always define what happens at n=0, n=1 |

---
## Mini-Quiz

In [ ]:
# Q1: What are the two requirements for a DP problem?
# Answer:

# Q2: What is the difference between top-down and bottom-up DP?
# Answer:

# Q3: You have a recursive function that is very slow.
# How do you check if memoization will help?
# Answer:

---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)